#### Model analysis

Diagnostics for one already trained experiment: decile precision and recall
by distance to development, and spatial maps of predictions and error.
Everything here loads from `results/models/<EXPERIMENT_NAME>/baseline/`,
written by `python scripts/gb_train.py baseline`, no retraining and no
rerun of the streamed fold inference that produced `fold_summary.csv` in
the first place. Supersedes the Model Analysis section of
`gradient_boosting.ipynb`.


In [ ]:
import os
import sys

sys.path.append(os.path.abspath("scripts"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from sklearn.metrics import precision_recall_curve

import experiments
import gb_common

pd.set_option("display.max_columns", None)


### Load the experiment

Set `EXPERIMENT_NAME` to whatever was passed to
`python scripts/gb_train.py baseline --experiment ...`. The fold models,
`fold_summary.csv`, and `oof_predictions.parquet` all come off disk, so this
cell is the only place retraining could happen, and it does not.


In [ ]:
EXPERIMENT_NAME = "neighborhood-features"

exp = experiments.get(EXPERIMENT_NAME)
baseline_dir = os.path.join(exp.results_dir, "baseline")
MODEL_ANALYSIS_DIR = os.path.join(exp.results_dir, "model_analysis")
os.makedirs(MODEL_ANALYSIS_DIR, exist_ok=True)

dataset_config = gb_common.load_dataset_config(exp.dataset_name)
models = [joblib.load(os.path.join(baseline_dir, f"model_fold{k}.joblib")) for k in range(gb_common.K_FOLDS)]
oof = pd.read_parquet(os.path.join(baseline_dir, "oof_predictions.parquet"))

# fold_summary.csv also carries a "mean" and "std" row sharing the same index
# column as the fold numbers, so the whole index reads back as strings, this
# recovers just the per fold rows with an integer index for .loc[fold_id, ...]
_raw_summary = pd.read_csv(os.path.join(baseline_dir, "fold_summary.csv"), index_col=0)
_raw_summary.index = _raw_summary.index.astype(str)
fold_summary = _raw_summary.loc[[str(k) for k in range(gb_common.K_FOLDS)]].copy()
fold_summary.index = fold_summary.index.astype(int)

con, _ = gb_common.connect_samples(exp.dataset_name)
fold_summary


### Precision and recall by distance to development decile

Deciles are cut once over the full 59M row table, every label, so the bin
edges are the same real distances for every fold. That matters here
specifically because precision and recall are threshold, count based
metrics, unlike average precision, so a per fold decile scheme would be
comparing different real distances between folds without saying so.

Conversions are lopsided across this binning before the model ever sees
them: the large majority of statewide `label == 1` pixels sit in the
closest one or two deciles, and by the ninth decile there are only a
handful of converters left in millions of pixels, zero beyond it. So
precision and recall out there are either a very noisy small count or fully
undefined, no positives to recall at all, and undefined bins are left off
the plot rather than drawn as a misleading zero.

Each fold classifies at its own best F1 threshold, the same operating point
`fold_summary`'s precision and recall columns already report, recovered
here on the raw out of fold score. Calibration is a monotone rescaling, so
it changes which number labels the threshold, not which pixels end up on
which side of it.


In [ ]:
DIST_DECILE_EDGES = con.execute(f"""
    SELECT {", ".join(f"QUANTILE_CONT({gb_common.DIST_COL}, {q / 10})" for q in range(1, 10))}
    FROM samples
""").df().iloc[0].tolist()

DIST_DECILE_LABELS = (
    [f"< {DIST_DECILE_EDGES[0]:,.0f} m"]
    + [f"{lo:,.0f} to {hi:,.0f} m" for lo, hi in zip(DIST_DECILE_EDGES[:-1], DIST_DECILE_EDGES[1:])]
    + [f"> {DIST_DECILE_EDGES[-1]:,.0f} m"]
)
DIST_DECILE_EDGES


In [ ]:
decile_rows = []
for fold_id in range(gb_common.K_FOLDS):
    sub = oof[oof["fold"] == fold_id]
    y_test = sub["label"].to_numpy()
    proba = sub["y_score"].to_numpy()
    dist = sub[gb_common.DIST_COL].to_numpy()

    # the same operating point fold_summary reports, recovered fresh here
    # since only the calibrated version of it survived into oof_predictions
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, proba)
    best_threshold, _, _, _ = gb_common.best_f1_operating_point(precision_curve, recall_curve, pr_thresholds)
    y_pred = proba >= best_threshold

    bin_idx = np.digitize(dist, DIST_DECILE_EDGES)
    for b in range(10):
        in_bin = bin_idx == b
        tp = int(np.sum(y_pred[in_bin] & (y_test[in_bin] == 1)))
        fp = int(np.sum(y_pred[in_bin] & (y_test[in_bin] == 0)))
        fn = int(np.sum(~y_pred[in_bin] & (y_test[in_bin] == 1)))
        decile_rows.append({
            "fold": fold_id, "decile": b, "label": DIST_DECILE_LABELS[b],
            "n": int(in_bin.sum()), "n_pos": tp + fn,
            "precision": tp / (tp + fp) if (tp + fp) > 0 else np.nan,
            "recall": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        })

decile_metrics = pd.DataFrame(decile_rows)
decile_metrics.pivot(index="label", columns="fold", values="n_pos").reindex(DIST_DECILE_LABELS)


### Figures

Axes are tightened to the real data range rather than the full 0 to 1: a 0
to 1 axis would flatten every fold into the bottom corner and hide exactly
the shape these figures exist to show. Adjust `ax.set_ylim` below if a
different experiment's recall or precision run higher than this one's.


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, ax = plt.subplots(figsize=(9, 6.4))
    for fold_id in range(gb_common.K_FOLDS):
        d = decile_metrics[decile_metrics["fold"] == fold_id]
        ax.plot(d["decile"], d["recall"], marker="o", ms=6, lw=2,
                color=gb_common.FOLD_COLORS[fold_id], label=f"fold {fold_id}")

    ax.set_xticks(range(10))
    ax.set_xticklabels(DIST_DECILE_LABELS, rotation=45, ha="right")
    ax.set_xlabel("Distance to 2019 development")
    ax.set_ylabel("Recall")
    ax.set_title(f"Recall by distance to development decile: {EXPERIMENT_NAME}")
    ax.legend(loc="upper right", frameon=False)
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.set_axisbelow(True)
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/recall_by_distance_decile.png")
    plt.show()


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, ax = plt.subplots(figsize=(9, 6.4))
    for fold_id in range(gb_common.K_FOLDS):
        d = decile_metrics[decile_metrics["fold"] == fold_id]
        ax.plot(d["decile"], d["precision"], marker="o", ms=6, lw=2,
                color=gb_common.FOLD_COLORS[fold_id], label=f"fold {fold_id}")

    ax.set_xticks(range(10))
    ax.set_xticklabels(DIST_DECILE_LABELS, rotation=45, ha="right")
    ax.set_xlabel("Distance to 2019 development")
    ax.set_ylabel("Precision")
    ax.set_title(f"Precision by distance to development decile: {EXPERIMENT_NAME}")
    ax.legend(loc="upper right", frameon=False)
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.set_axisbelow(True)
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/precision_by_distance_decile.png")
    plt.show()


### Spatial diagnostics

The decile breakdown above answers whether skill is distance dependent. It
says nothing about where in Florida that skill actually lands, and that is
the question a reader looking at a map of their county actually has.

All three maps below come from one out of fold pass: every pixel is scored
by the model from the fold that held it out, so together the five folds
cover the whole state exactly once, with no pixel ever scored by a model
that trained on it. This is a fresh, deterministic spatial subsample
(`gb_common.sample_fold_with_xy`, a different draw than
`oof_predictions.parquet`), not the full out of fold table, since plotting
59M points is not the point of a diagnostic map: 100,000 pixels per fold,
about 500,000 statewide, is enough to trace the coastline and the
conversion clusters without the renderer or the saved PNG choking on it.

Each fold's predicted positive uses that fold's own best F1 threshold,
recovered from `fold_summary` rather than refit here, applied to the
calibrated score.


In [ ]:
MAP_SAMPLE_ROWS_PER_FOLD = 100_000  # about 500k points statewide, dense enough for a slide sized map

map_frames = []
for fold_id in range(gb_common.K_FOLDS):
    log_mean, log_std = gb_common.fit_dist_normalizer(con, fold_id)
    df = gb_common.sample_fold_with_xy(con, fold_id, MAP_SAMPLE_ROWS_PER_FOLD, log_mean, log_std)
    proba = models[fold_id].predict_proba(df[exp.feature_cols])[:, 1]
    df["score"] = gb_common.calibrate(proba, dataset_config["pos_weight_mult"])
    df["y_pred"] = (df["score"] >= fold_summary.loc[fold_id, "best_threshold"]).astype(int)
    map_frames.append(df[["x", "y", "label", gb_common.TARGET_COL, "score", "y_pred"]])
    del df, proba

map_predictions = pd.concat(map_frames, ignore_index=True)
del map_frames

# tag each pixel once so the error map below doesn't recompute this per panel
map_predictions["outcome"] = np.select(
    [
        (map_predictions[gb_common.TARGET_COL] == 1) & (map_predictions["y_pred"] == 1),
        (map_predictions[gb_common.TARGET_COL] == 0) & (map_predictions["y_pred"] == 1),
        (map_predictions[gb_common.TARGET_COL] == 1) & (map_predictions["y_pred"] == 0),
    ],
    ["true_positive", "false_positive", "false_negative"],
    default="true_negative",
)

len(map_predictions), map_predictions[gb_common.TARGET_COL].mean(), map_predictions["y_pred"].mean()


#### Labels versus predictions

Same axes, same point styling, side by side so the eye does the comparing
instead of a legend. Gray covers everything that is not a developed
conversion, either the true label or the model's call depending on the
panel; color marks "converted to developed." If the model's orange cloud on
the right traces the same shape as the true one on the left, the skill in
the decile plot above is a real spatial pattern, not an artifact of a few
lucky folds.


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13, 7.2), sharex=True, sharey=True)

    panels = [(gb_common.TARGET_COL, "True conversions, 2019 to 2024"), ("y_pred", "Model predicted conversions")]
    for ax, (col, title) in zip(axes, panels):
        neg = map_predictions[map_predictions[col] == 0]
        pos = map_predictions[map_predictions[col] == 1]
        ax.scatter(neg["x"], neg["y"], s=1, alpha=0.15, color="#BBBBBB", zorder=2,
                   label="Stayed wetland or converted to non developed")
        ax.scatter(pos["x"], pos["y"], s=4, alpha=0.75, color="#D55E00", zorder=3,
                   label="Converted to developed")
        ax.set_aspect("equal")
        ax.set_title(title)
        ax.set_xlabel("Easting (m, NLCD Albers)")
        ax.tick_params(axis="x", rotation=30)

    axes[0].set_ylabel("Northing (m, NLCD Albers)")
    axes[0].legend(markerscale=10, loc="upper left", frameon=False, fontsize=10)
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/labels_vs_predictions_map.png")
    plt.show()


#### Errors: false positives and false negatives

A false positive is a pixel the model flags as converting that stayed
wetland; a false negative is a real conversion the model missed. Split into
separate panels because they tend to sit in different places for different
reasons: a false positive is usually the model reading a "next to
development already" signal that never actually flips within the window,
while a false negative is a conversion the embeddings gave little signal
for at all. Both panels share the same gray backdrop as above for
geographic context.


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13, 7.2), sharex=True, sharey=True)

    background = map_predictions[map_predictions["outcome"] == "true_negative"]
    error_panels = [
        (map_predictions[map_predictions["outcome"] == "false_positive"], "#0072B2", "False positives"),
        (map_predictions[map_predictions["outcome"] == "false_negative"], "#CC79A7", "False negatives"),
    ]
    for ax, (err, color, title) in zip(axes, error_panels):
        ax.scatter(background["x"], background["y"], s=1, alpha=0.1, color="#BBBBBB", zorder=2)
        ax.scatter(err["x"], err["y"], s=5, alpha=0.8, color=color, zorder=3)
        ax.set_aspect("equal")
        ax.set_title(f"{title} (n={len(err):,})")
        ax.set_xlabel("Easting (m, NLCD Albers)")
        ax.tick_params(axis="x", rotation=30)

    axes[0].set_ylabel("Northing (m, NLCD Albers)")
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/error_map.png")
    plt.show()


#### Model score across every 2019 wetland pixel

Not thresholded. The two maps above collapse a continuous score to a single
yes or no call, which hides how confident the model is anywhere short of
the operating point. Color here is the calibrated probability of
conversion on a log scale: a well under 1% base rate means almost every
pixel's score sits under 1%, and a linear color scale would flatten the
entire state to one shade. Negatives are included, since the point of this
map is the full risk surface, not just where converters happened to sit.


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, ax = plt.subplots(figsize=(10.5, 7.5))

    plot_df = map_predictions.sort_values("score")  # lowest score first, so the hottest pixels draw on top
    vmin = max(plot_df.loc[plot_df["score"] > 0, "score"].min(), 1e-5)
    vmax = plot_df["score"].max()

    sc = ax.scatter(plot_df["x"], plot_df["y"], c=plot_df["score"].clip(lower=vmin),
                    s=3, cmap="viridis", norm=LogNorm(vmin=vmin, vmax=vmax), zorder=2)
    ax.set_aspect("equal")
    ax.set_xlabel("Easting (m, NLCD Albers)")
    ax.set_ylabel("Northing (m, NLCD Albers)")
    ax.tick_params(axis="x", rotation=30)
    ax.set_title(f"Calibrated conversion score across 2019 wetlands: {EXPERIMENT_NAME}", fontsize=13)
    cbar = fig.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("Model score: calibrated probability of conversion to developed (log scale)")
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/score_map.png")
    plt.show()


### Feature importance

Which of the model's input columns actually earned their place. Read off
the fold whose average precision is best among the five, in
`fold_summary`, since importance is a property of one trained model, not
something that averages cleanly across five separately fit trees.

Gain, not split count: gain adds up how much each split on a feature
actually reduced the training loss, while a plain split count would credit
a cheap feature that gets used often but barely moves the loss the same as
one that rarely splits but matters a great deal each time it does. LightGBM
keeps both on the trained booster, so this reads gain off the model
directly, no retraining needed.


In [ ]:
best_fold = int(fold_summary["avg_precision"].idxmax())
best_model = models[best_fold]

TOP_N = 20
gain = pd.Series(best_model.booster_.feature_importance(importance_type="gain"), index=exp.feature_cols)
top_gain = gain.sort_values(ascending=False).head(TOP_N)

with plt.rc_context(gb_common.SLIDE_RC):
    fig, ax = plt.subplots(figsize=(9, 0.42 * TOP_N + 1.2))
    ax.barh(top_gain.index[::-1], top_gain.values[::-1], color="#2a78d6")
    ax.set_xlabel("Total gain (training loss reduction from splits on this feature)")
    ax.set_title(f"Top {TOP_N} feature importances, fold {best_fold} model: {EXPERIMENT_NAME}")
    fig.tight_layout()
    fig.savefig(f"{MODEL_ANALYSIS_DIR}/feature_importance.png")
    plt.show()
